In [2]:
"""Utility functions for training and evaluation"""

import torch
import numpy as np
import random

def set_seed(seed=42):
    """Fix random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

def rmse(y_pred, y_true):
    """Root Mean Square Error"""
    return torch.sqrt(torch.mean((y_pred - y_true) ** 2)).item()

def mae(y_pred, y_true):
    """Mean Absolute Error"""
    return torch.mean(torch.abs(y_pred - y_true)).item()

def mape(y_pred, y_true, eps=1e-6):
    """
    Mean Absolute Percentage Error
    Uses eps to avoid division by zero
    """
    # Only consider non-zero true values
    mask = torch.abs(y_true) > 0.1
    if mask.sum() == 0:
        return 0.0
    
    absolute_percentage_error = torch.abs((y_pred - y_true)[mask] / (y_true[mask] + eps))
    return (absolute_percentage_error.mean()).item() * 100

def smape(y_pred, y_true, eps=1e-6):
    """Symmetric Mean Absolute Percentage Error"""
    denominator = (torch.abs(y_pred) + torch.abs(y_true) + eps) / 2
    return torch.mean(torch.abs(y_pred - y_true) / denominator).item() * 100

def calculate_all_metrics(y_pred, y_true):
    """Calculate all metrics at once"""
    return {
        'RMSE': rmse(y_pred, y_true),
        'MAE': mae(y_pred, y_true),
        'MAPE': mape(y_pred, y_true),
        'sMAPE': smape(y_pred, y_true)
    }

def inverse_transform_predictions(predictions, targets, scaler, target_idx=0):
    """
    Inverse transform normalized predictions back to original scale
    """
    # Create dummy array with same shape as original features
    dummy = np.zeros((predictions.shape[0], predictions.shape[1], scaler.mean_.shape[0]))
    dummy[:, :, target_idx] = predictions
    dummy_targets = np.zeros_like(dummy)
    dummy_targets[:, :, target_idx] = targets
    
    # Inverse transform
    pred_original = scaler.inverse_transform(dummy.reshape(-1, scaler.mean_.shape[0]))
    target_original = scaler.inverse_transform(dummy_targets.reshape(-1, scaler.mean_.shape[0]))
    
    return pred_original[:, target_idx].reshape(predictions.shape), target_original[:, target_idx].reshape(targets.shape)

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            return False
        
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
            return False